# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Run on Kaggle. Upload or clone this repo first; data is supplied separately via Kaggle Input.
%pip -q install timm thop matplotlib openpyxl
import sys, platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO_DIR = Path("/kaggle/working/K4-DAY02-TongTranTienDung-2A202602791")
CODE_DIR = REPO_DIR / "submissions/2A202602791_Tong_Tran_Tien_Dung/code"
assert CODE_DIR.is_dir(), f"Set REPO_DIR to the uploaded/cloned repository: {CODE_DIR}"
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(CODE_DIR))
import dataset, model, losses, train
print("python", platform.python_version(), "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

### Dữ liệu Kaggle Input
Gắn dataset ảnh và bốn file CSV vào notebook. Đặt hai đường dẫn bên dưới đến thư mục chứa ảnh và thư mục chứa CSV. Notebook không tự tải dữ liệu.


In [ ]:
# Edit these two paths to match your Kaggle Input folder.
IMAGES_DIR = Path("/kaggle/input/deepweeds/images")
LABELS_DIR = Path("/kaggle/input/deepweeds/labels")
assert IMAGES_DIR.is_dir(), f"Image directory not found: {IMAGES_DIR}"
assert LABELS_DIR.is_dir(), f"Labels directory not found: {LABELS_DIR}"
print("Image examples:", [p.name for p in list(IMAGES_DIR.glob("*.jpg"))[:5]])
print("CSV files:", sorted(p.name for p in LABELS_DIR.glob("*subset0.csv")))

In [ ]:
# Confirm paths before running the split audit.
print("IMAGES_DIR =", IMAGES_DIR)
print("LABELS_DIR =", LABELS_DIR)

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# Official fold 0 only. check_split aborts on overlap, missing files or wrong total.
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_report = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)
counts = pd.DataFrame(split_report["per_class"]).reindex(range(dataset.NUM_CLASSES))
counts.index = dataset.CLASS_NAMES
display(counts)
ax = counts.plot.bar(figsize=(12, 4), title="DeepWeeds fold 0: class distribution")
ax.set(xlabel="Class", ylabel="Images")
plt.tight_layout()
plt.show()
totals = counts.sum(axis=1)
print("Largest/smallest class ratio:", round(totals.max() / totals.min(), 2))
print("Paper reference only: Negative 9,106; weed classes 1,009–1,125 each.")
print("Your actual totals:", totals.to_dict())

# Three sample images per class, from TRAIN only.
from PIL import Image
fig, axes = plt.subplots(dataset.NUM_CLASSES, 3, figsize=(9, 22))
for label in range(dataset.NUM_CLASSES):
    rows = train_df.loc[train_df["Label"] == label, "Filename"].head(3)
    assert len(rows) == 3, f"Class {label} has fewer than three training images"
    for col, filename in enumerate(rows):
        with Image.open(IMAGES_DIR / filename) as image:
            axes[label, col].imshow(image.convert("RGB"))
        axes[label, col].set_title(f"{dataset.CLASS_NAMES[label]}: {filename}", fontsize=8)
        axes[label, col].axis("off")
plt.tight_layout()
plt.show()

# Check sizes and channels on a deterministic sample of 100 TRAIN images.
sample = train_df.sample(n=min(100, len(train_df)), random_state=0)
shapes = []
for filename in sample["Filename"]:
    with Image.open(IMAGES_DIR / filename) as image:
        shapes.append((image.width, image.height, len(image.getbands())))
print("Sample (width, height, channels) counts:")
print(pd.Series(shapes).value_counts())

In [ ]:
# Pipeline smoke check. It uses a tiny fixed TRAIN batch, never val or test for optimization.
train.set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sample_df = train_df.head(8)
plain = dataset.DeepWeedsDataset(sample_df, IMAGES_DIR, dataset.build_transforms(False))
x = torch.stack([plain[i][0] for i in range(4)]).to(device)
y = torch.tensor([plain[i][1] for i in range(4)], device=device)
net = model.build_model("resnet18", pretrained=False, init="scratch").to(device)
criterion = losses.build_criterion("ce")
net.eval()
with torch.inference_mode():
    initial_loss = criterion(net(x), y).item()
print("Initial CE:", initial_loss, "; ln(9):", np.log(9))
print("Difference from ln(9) is diagnostic, not a strict assertion.")

# Visualize augmented TRAIN samples after denormalization.
aug_ds = dataset.DeepWeedsDataset(sample_df, IMAGES_DIR, dataset.build_transforms(True, 224, "basic"))
mean = torch.tensor(dataset.IMAGENET_MEAN)[:, None, None]
std = torch.tensor(dataset.IMAGENET_STD)[:, None, None]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    image, label, filename = aug_ds[i]
    ax.imshow((image * std + mean).clamp(0, 1).permute(1, 2, 0))
    ax.set_title(f"{dataset.CLASS_NAMES[label]}\n{filename}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

# Fit the exact same tiny batch until nearly zero loss; stop early when it succeeds.
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=0)
net.train()
print("Training mode:", net.training)
for step in range(250):
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(net(x), y)
    loss.backward()
    optimizer.step()
    if loss.item() < 0.05:
        break
net.eval()
with torch.inference_mode():
    final_loss = criterion(net(x), y).item()
print("Tiny-batch overfit:", {"steps": step + 1, "train_loss": loss.item(), "eval_loss": final_loss})
assert loss.item() < 0.05, "Tiny batch did not overfit; inspect data, optimizer and model"
print("Evaluation mode:", not net.training)

## Bước 1 — So sánh backbone (ít nhất 5)

Chạy cùng cấu hình T00, fold 0 và seed 0. Chỉ thay `exp_id` và `backbone`. Mỗi kết quả được ghi ngay vào sheet **Backbones** để có thể tiếp tục sau khi Kaggle ngắt phiên. Không chạy test ở bước này. Số đo độ trễ là forward batch 1, có warmup và đồng bộ CUDA, không tính tiền xử lý.

In [ ]:
import json
from dataclasses import asdict
import timm
import benchmark
assert torch.cuda.is_available(), "Enable a Kaggle GPU before running five full experiments"
from train import Config, run, run_dir, SUBMISSION_DIR

# One model from every required family, with ConvNeXt and a lightweight model.
BACKBONES = [
    ("B01", "resnet50"),
    ("B02", "resnext50_32x4d"),
    ("B03", "convnext_tiny"),
    ("B04", "deit_small_patch16_224"),
    ("B05", "efficientnet_b0"),
]
SEED = 0
results_path = SUBMISSION_DIR / "results.xlsx"
rows = []

for exp_id, backbone in BACKBONES:
    cfg = Config(
        exp_id=exp_id, backbone=backbone, seed=SEED,
        images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR),
        save_test_predictions=False,
    )
    folder = run_dir(cfg)
    summary_path = folder / "summary.json"
    if summary_path.exists() and (folder / "best.pt").exists():
        saved_cfg = json.loads((folder / "config.json").read_text(encoding="utf-8"))
        if saved_cfg != asdict(cfg):
            raise ValueError(f"{exp_id}: saved configuration differs from T00; use a new exp_id")
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        print(f"{exp_id}: using completed training log")
    else:
        summary = run(cfg)

    latency_path = folder / "latency_batch1.json"
    gpu_name = torch.cuda.get_device_name(0)
    cached_latency = (json.loads(latency_path.read_text(encoding="utf-8"))
                      if latency_path.exists() else None)
    if (cached_latency and cached_latency.get("gpu") == gpu_name and
        cached_latency.get("img_size") == cfg.img_size and
        cached_latency.get("dtype") == "amp" and
        latency_path.stat().st_mtime >= (folder / "best.pt").stat().st_mtime):
        latency = cached_latency
    else:
        device_name = "cuda" if torch.cuda.is_available() else "cpu"
        restored = model.build_model(backbone, pretrained=False, init="scratch",
                                     drop_rate=cfg.drop_rate)
        state = torch.load(folder / "best.pt", map_location="cpu", weights_only=False)
        model.load_checkpoint_state(restored, state["state_dict"])
        latency = benchmark.latency_report(
            restored, batch_size=1, img_size=cfg.img_size,
            dtype="amp" if device_name == "cuda" else "fp32",
            device=device_name, warmup=10, iters=50,
        )
        latency_path.write_text(json.dumps(latency, indent=2), encoding="utf-8")
        del restored
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    row = {
        "exp_id": exp_id,
        "backbone": backbone,
        "weight_tag": summary["weight_tag"],
        "timm_version": timm.__version__,
        "params_m": summary["params_m"],
        "gmacs_thop": summary["gmacs"],
        "img_size": cfg.img_size,
        "epochs": cfg.epochs,
        "seed": cfg.seed,
        "macro_f1_val": summary["macro_f1_val"],
        "top1_val": summary["top1_val"],
        "train_seconds_per_epoch": summary["train_seconds_per_epoch"],
        "latency_batch1_p50_ms": latency["p50"],
        "latency_batch1_p95_ms": latency["p95"],
        "latency_dtype": latency["dtype"],
        "latency_device": latency["gpu"],
        "curve": str(SUBMISSION_DIR / "curves" /
                     f"{exp_id}_{backbone}_seed{SEED}.png"),
        "note": "T00 common recipe; one seed; val only",
    }
    rows.append(row)
    frame = pd.DataFrame(rows)
    if results_path.exists():
        with pd.ExcelWriter(results_path, engine="openpyxl", mode="a",
                            if_sheet_exists="replace") as writer:
            frame.to_excel(writer, sheet_name="Backbones", index=False)
    else:
        with pd.ExcelWriter(results_path, engine="openpyxl") as writer:
            frame.to_excel(writer, sheet_name="Backbones", index=False)
    display(frame.tail(1))

backbone_results = pd.DataFrame(rows).sort_values("macro_f1_val", ascending=False)
display(backbone_results)
print("Best val macro-F1:", backbone_results.iloc[0][["exp_id", "backbone", "macro_f1_val"]].to_dict())
fastest = backbone_results.sort_values("latency_batch1_p50_ms").iloc[0]
print("Lowest batch-1 latency:", fastest[["exp_id", "backbone", "latency_batch1_p50_ms"]].to_dict())
if backbone_results["gmacs_thop"].isna().any():
    print("Some GMAC measurements are unavailable; inspect each run log and THOP support.")
print("Comparison uses one seed. Choose 1–2 backbones after examining both F1 and latency.")
pareto = backbone_results.loc[
    [not any((other["macro_f1_val"] >= item["macro_f1_val"] and
              other["latency_batch1_p50_ms"] <= item["latency_batch1_p50_ms"] and
              (other["macro_f1_val"] > item["macro_f1_val"] or
               other["latency_batch1_p50_ms"] < item["latency_batch1_p50_ms"]))
             for _, other in backbone_results.iterrows())
     for _, item in backbone_results.iterrows()]
]
print("Models on the F1/latency Pareto frontier:")
display(pareto[["exp_id", "backbone", "macro_f1_val", "latency_batch1_p50_ms"]])

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(backbone_results["latency_batch1_p50_ms"], backbone_results["macro_f1_val"])
for _, item in backbone_results.iterrows():
    ax.annotate(item["exp_id"] + " " + item["backbone"],
                (item["latency_batch1_p50_ms"], item["macro_f1_val"]), xytext=(5, 5),
                textcoords="offset points", fontsize=8)
ax.set(xlabel="Batch-1 forward p50 (ms)", ylabel="Macro-F1 val",
       title="Backbone quality and latency (T00, seed 0)")
fig.tight_layout()
tradeoff_path = SUBMISSION_DIR / "curves" / "backbone_tradeoff.png"
tradeoff_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(tradeoff_path, dpi=160)
plt.show()

## Bước 2 — Công thức huấn luyện

Chọn backbone từ sheet Backbones sau khi xem kết quả Bước 1. T00 là công thức nền. T01–T06 thay đúng một trục A, B hoặc C; T07 kết hợp biến thể tốt nhất đã thử ở B và C. Mỗi cấu hình giữ nguyên fold, seed và số epoch. Kết quả vòng sàng chỉ có một seed, nên không kết luận từ chênh lệch nhỏ.

In [ ]:
import workflow
SELECTED_BACKBONE = str(backbone_results.iloc[0]["backbone"])  # Edit after reviewing Step 1.
print("Backbone chosen from val:", SELECTED_BACKBONE)
training_results = workflow.run_ablation(SELECTED_BACKBONE, IMAGES_DIR, LABELS_DIR, seed=0)
display(training_results.sort_values("macro_f1_val", ascending=False))
selected_training = workflow.best_training_config(training_results)
print("Best training recipe on val:", selected_training.exp_id)
print("Review Training sheet, rare-class F1 and one-seed uncertainty before accepting this choice.")

## Bước 3 — Suy luận và độ trễ

Dùng checkpoint tốt nhất từ Bước 2. So sánh I00 một view với I01 lật ngang, I02 năm crop, I03 gộp logit và I07 temperature scaling trên **val**. Mỗi phương pháp được đo batch 1 và batch 16: 10 lần warmup, 50 lần đo, đồng bộ CUDA, FP32; phép biến đổi trên GPU được tính, đọc ảnh từ đĩa không được tính.

In [ ]:
inference_results, recommendation = workflow.compare_inference(selected_training, large_batch=16)
display(inference_results.sort_values("macro_f1_val", ascending=False))
print("Val-only suggestion:", recommendation)
FINAL_METHOD = recommendation["method"]       # I00, I01, I02 or I03; edit after reviewing latency.
USE_CALIBRATION = recommendation["calibrate"] # Refit one T on val for each final seed.
print("Inspect Inference and Latency sheets before fixing the final method.")

## Bước 4 — Chốt trên val, sau đó mới đọc test

Xem bảng Backbones, Training và Inference trước khi bật hai cờ bên dưới. Bước huấn luyện chung kết chạy F01 và mốc T00 cùng seed 0, 1, 2, không gộp val. Cờ đọc test mặc định tắt. Mỗi file test được ghi đúng một lần; nếu quá trình dừng giữa chừng sau khi đã bắt đầu đọc test, hãy kiểm tra dấu `test_started.json` thay vì chạy lại tự động.

In [ ]:
from dataclasses import asdict
from dataclasses import replace
from eval import read_pred, check_against_csv

FINAL_RECIPE = selected_training   # Edit only using val evidence before enabling final training.
FINAL_METHOD = FINAL_METHOD         # Chosen from val in Step 3.
USE_CALIBRATION = USE_CALIBRATION
RUN_FINAL_TRAINING = False          # Set True after reviewing all val results.
RUN_FINAL_TEST = False              # Set True only after training and fixing the configuration.
SEEDS = (0, 1, 2)
selection_file = SUBMISSION_DIR / "final_selection.json"
selection = {"recipe": asdict(FINAL_RECIPE), "method": FINAL_METHOD,
             "calibrate": USE_CALIBRATION, "seeds": list(SEEDS)}

if RUN_FINAL_TRAINING:
    if selection_file.exists():
        previous = json.loads(selection_file.read_text(encoding="utf-8"))
        if previous != selection:
            raise ValueError("Final selection changed. Inspect the saved selection before proceeding.")
    else:
        selection_file.write_text(json.dumps(selection, indent=2), encoding="utf-8")
    final_training = workflow.train_final_pair(FINAL_RECIPE, SEEDS)
    display(final_training)
else:
    print("Final training is paused. Review val results, then set RUN_FINAL_TRAINING=True.")

if RUN_FINAL_TEST:
    if not RUN_FINAL_TRAINING:
        raise ValueError("Train and lock the final configuration before reading test")
    for seed in SEEDS:
        final_cfg = replace(FINAL_RECIPE, exp_id="F01", seed=seed,
                            save_test_predictions=False)
        baseline_cfg = train.Config(exp_id="T00", backbone=FINAL_RECIPE.backbone,
                                    seed=seed, images_dir=str(IMAGES_DIR),
                                    labels_dir=str(LABELS_DIR),
                                    save_test_predictions=False)
        print("Test:", workflow.predict_test_once(final_cfg, FINAL_METHOD, USE_CALIBRATION))
        print("Baseline test:", workflow.predict_test_once(baseline_cfg, "I00", False))
else:
    print("Test remains unopened. Set RUN_FINAL_TEST=True only after the val-based choice is fixed.")

In [ ]:
# Recompute official metrics from saved prediction files only; no model or test loader is run here.
import subprocess
pred_dir = SUBMISSION_DIR / "predictions"
eval_out = SUBMISSION_DIR / "eval_out"
for exp_id in ("F01", "T00"):
    files = [pred_dir / f"{exp_id}_seed{seed}_test.csv" for seed in SEEDS]
    if not all(path.is_file() for path in files):
        raise FileNotFoundError(f"Finish Step 4 before scoring {exp_id}: {files}")
    command = [sys.executable, str(REPO_DIR / "eval.py"), "score",
               "--pred", *map(str, files),
               "--test-csv", str(LABELS_DIR / "test_subset0.csv"),
               "--tag", exp_id, "--out", str(eval_out)]
    labels_csv = LABELS_DIR / "labels.csv"
    if labels_csv.exists():
        command += ["--labels", str(labels_csv)]
    subprocess.run(command, check=True)

In [ ]:
# Official rubric check. This reads saved CSVs; it never calls the model.
command = [sys.executable, str(REPO_DIR / "eval.py"), "grade",
           "--final", *[str(pred_dir / f"F01_seed{seed}_test.csv") for seed in SEEDS],
           "--baseline", *[str(pred_dir / f"T00_seed{seed}_test.csv") for seed in SEEDS],
           "--final-val", *[str(pred_dir / f"F01_seed{seed}_val.csv") for seed in SEEDS],
           "--test-csv", str(LABELS_DIR / "test_subset0.csv"),
           "--val-csv", str(LABELS_DIR / "val_subset0.csv"),
           "--out", str(eval_out)]
if USE_CALIBRATION:
    command += ["--uncal", *[str(pred_dir / f"F01uncal_seed{seed}_test.csv") for seed in SEEDS]]
if (LABELS_DIR / "labels.csv").exists():
    command += ["--labels", str(LABELS_DIR / "labels.csv")]
subprocess.run(command, check=True)

## Bước 5 — Bảng kết quả và báo cáo

Sau khi eval.py xác nhận các file dự đoán, tạo đủ bảy sheet trong `results.xlsx`, ma trận nhầm lẫn, danh sách ảnh dự đoán sai và bản thảo `report.md` từ số liệu thật. Đọc lại báo cáo và bổ sung nhận xét định tính, phần cứng, phiên bản thư viện và link Kaggle notebook.

In [ ]:
artifact_summary = workflow.make_final_artifacts(FINAL_RECIPE, FINAL_METHOD,
                                                   USE_CALIBRATION, SEEDS)
display(pd.read_excel(SUBMISSION_DIR / "results.xlsx", sheet_name="Summary"))
display(pd.read_excel(SUBMISSION_DIR / "results.xlsx", sheet_name="Final"))
print("Created from real predictions:", artifact_summary)
print("Review report.md and add data interpretation and the Kaggle notebook link.")